# 💘 Valentine's Day Hackathon 2026

**The Challenge:** Given a person's demographic and social attributes, predict whether they have a Valentine's date (`Has_Valentine = 1`) or not (`Has_Valentine = 0`).

**Submit here:** [Kaggle Competition](https://www.kaggle.com/t/2ad09a2d970241e88bbd69e469819448)



This notebook will walk you through the full machine learning pipeline:

1. **Setup & Data Loading**
2. **Exploratory Data Analysis (EDA)** - understand your data before modelling
3. **Feature Engineering** - create new features to help the model
4. **Preprocessing** - prepare data for ML algorithms
5. **Modelling** - train models to make predictions
6. **Cross Validation** - evaluate your model properly
7. **Hyperparameter Tuning** - squeeze out better performance
8. **Generate Submission**

The notebook runs **start to finish** and produces a valid submission file. But the baseline is intentionally simple — your job is to improve it! Look for the **🔧 YOUR TURN** cells.

## 1. Setup & Data Loading

In [ ]:
# If you need to install anything, uncomment and run:
# !pip install pandas numpy scikit-learn matplotlib seaborn lightgbm xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print('All imports loaded')

In [ ]:
# Update these paths if your files are in a different folder
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train: {train.shape[0]} rows, {train.shape[1]} columns')
print(f'Test:  {test.shape[0]} rows, {test.shape[1]} columns')

## 2. Exploratory Data Analysis (EDA)

Before building any model, you need to **understand your data**. What columns do we have? What's missing? What patterns exist?

EDA isn't just a formality, the insights you find here directly inform what features you create and which models you choose.

In [ ]:
# First look at the data
train.head()

In [ ]:
# Column types and non-null counts
train.info()

In [ ]:
# Summary statistics for numeric columns
train.describe()

In [ ]:
# How balanced is our target? If it's very lopsided (e.g. 99% vs 1%),
# we'd need special handling. Let's check:
fig, ax = plt.subplots(figsize=(5, 3))
train['Has_Valentine'].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('Target Distribution')
ax.set_xticklabels(['No Valentine (0)', 'Has Valentine (1)'], rotation=0)
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f"\nTarget split:")
print(train['Has_Valentine'].value_counts(normalize=True).round(4))

In [ ]:
# Missing values - this dataset has a LOT of them, unlike some of the previous datasets from Playground competitions.
# How you handle missing data can make or break your model.
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
missing_pct[missing_pct > 0].plot(kind='barh', ax=ax, color='#3498db')
ax.set_xlabel('% Missing')
ax.set_title('Missing Values by Column')
plt.tight_layout()
plt.show()

In [ ]:
# Which numeric features correlate with the target?
# This tells us which features might be useful for prediction.
numeric_cols = train.select_dtypes(include='number').columns.drop('Id')
correlations = train[numeric_cols].corr()['Has_Valentine'].drop('Has_Valentine').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
correlations.plot(kind='barh', ax=ax, color=correlations.apply(lambda x: '#2ecc71' if x > 0 else '#e74c3c'))
ax.set_title('Correlation with Has_Valentine')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# How do categorical features relate to the target?
cat_cols = ['Gender', 'Education', 'Job_Type', 'Location_Type']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, cat_cols):
    rates = train.groupby(col)['Has_Valentine'].mean().sort_values(ascending=False)
    rates.plot(kind='bar', ax=ax, color='#9b59b6')
    ax.set_title(f'{col} vs Has_Valentine')
    ax.set_ylabel('% with Valentine')
    ax.set_ylim(0.4, 0.6)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### YOUR TURN - Explore some more
There are more columns we haven't looked at yet. What about `Zodiac_Sign`, `Pets`, `Favorite_Color`, `Social_Media_Presence`? Do any of them matter?

In [ ]:
# Your EDA here


## 3. Feature Engineering

Raw data is rarely enough. **Feature engineering** means creating new columns that help the model find patterns it couldn't see before.

For example, `Height_cm` and `Weight_kg` alone are okay - but **BMI** (which combines them) might be more meaningful.

We apply the same transformations to both train and test so they have matching columns.

In [ ]:
def engineer_features(df):
    """Create new features. Applied to both train and test."""
    df = df.copy()
    
    # --- Example 1: BMI from height and weight ---
    df['BMI'] = df['Weight_kg'] / ((df['Height_cm'] / 100) ** 2)
    
    # --- Example 2: Income relative to age ---
    # A 25-year-old earning 60k is different from a 55-year-old earning 60k
    df['Income_per_Age'] = df['Income'] / (df['Age'] + 1)
    
    # --- Example 3: Missingness as a feature ---
    # Sometimes the FACT that data is missing tells you something.
    # Someone who didn't report their income might be different from someone who did.
    df['n_missing'] = df.isnull().sum(axis=1)
    
    # TO DO - Add more features below!
    
    # Idea: What if you multiply Appearance_Score by Social_Skills_Score?
    # df['Appear_x_Social'] = ...
    
    # Idea: What about interactions with Dating_App_User?
    # df['Dating_x_Extra'] = ...
    
    # Idea: Does the Survey_Date contain useful info? (month, day of week, hour?)
    # df['Survey_Date'] = pd.to_datetime(df['Survey_Date'], errors='coerce')
    # df['Survey_Month'] = ...
    
    # Idea: Composite score from multiple numeric features?
    
    # Idea: Missingness indicators for specific important columns?
    
    return df

train = engineer_features(train)
test  = engineer_features(test)

print(f'Train columns: {train.shape[1]}  |  Test columns: {test.shape[1]}')

## 4. Preprocessing

ML models need numbers. Our categorical columns (`Gender`, `Education`, etc.) are text, so we need to encode them.

**LabelEncoder** turns each unique category into an integer (e.g. Male=0, Female=1, Non-Binary=2). We fit it on the combined train+test data so both get consistent encodings.

In [ ]:
cat_cols = ['Gender', 'Education', 'Job_Type', 'Social_Media_Presence',
            'Location_Type', 'Zodiac_Sign', 'Pets', 'Favorite_Color']

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

print('Categorical columns encoded')
print(f'Example — Gender values: {sorted(train["Gender"].unique())}')

In [ ]:
# Separate features (X) from the target (y)
drop_cols = ['Id', 'Has_Valentine', 'Survey_Date']
feature_cols = [c for c in train.columns if c not in drop_cols]

X      = train[feature_cols]
y      = train['Has_Valentine']
X_test = test[feature_cols]
test_ids = test['Id']

print(f'Features: {len(feature_cols)}')
print(f'X: {X.shape}  |  y: {y.shape}  |  X_test: {X_test.shape}')

## 5. Modelling

Now we train a model to learn the relationship between our features and the target.

We'll start with **Logistic Regression**, a simple, interpretable model that's a great baseline. It draws a line (well, a hyperplane) to separate the two classes.

### 5a. Logistic Regression (Baseline)

In [ ]:
# Logistic Regression needs scaled features and no NaNs.
# We'll fill missing values with the median and scale everything.

# Fill NaNs with median (simple strategy - you can do better!)
X_filled      = X.fillna(X.median())
X_test_filled = X_test.fillna(X.median())  # use TRAIN medians for test

# Scale features to mean=0, std=1 (important for logistic regression)
scaler = StandardScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X_filled), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_filled), columns=feature_cols)

print('Data preprocessed for Logistic Regression')

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_scaled, y)

# Predict probabilities (not just 0/1 - we need probabilities for ROC-AUC)
lr_test_preds = lr_model.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression trained')
print(f'Prediction range: {lr_test_preds.min():.3f} to {lr_test_preds.max():.3f}')

## 6. Cross Validation

How good is our model? We can't just check accuracy on the training data the model has already seen it!

**Cross validation** splits the training data into folds, trains on some folds, and evaluates on the held-out fold. This gives us an honest estimate of how the model will perform on unseen data.

We use **StratifiedKFold** to make sure each fold has the same ratio of 0s and 1s.

In [ ]:
# 5-fold stratified cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled, y,
    cv=cv,
    scoring='roc_auc'
)

print('Logistic Regression CV Results:')
print(f'  Fold AUCs: {[f"{s:.4f}" for s in lr_cv_scores]}')
print(f'  Mean AUC:  {lr_cv_scores.mean():.4f} ± {lr_cv_scores.std():.4f}')
print(f'\nThis is your baseline to beat!')

YOUR TURN - Try Different Models

Logistic Regression is a **linear** model — it can only learn straight-line decision boundaries. Can you do better with a non-linear model?

Here are some ideas to try. Each cell has a skeleton to get you started.

### 5b. Random Forest

Random Forests build many decision trees on random subsets of the data and average their predictions. They handle missing values and non-linear relationships well.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forests can handle NaNs if you fill them, and don't need scaling.
# You can use X_filled (not X_scaled) here.

# rf_model = RandomForestClassifier(
#     n_estimators=???,   # how many trees? try 100, 300, 500
#     max_depth=???,      # how deep? try None, 10, 20
#     random_state=42,
#     n_jobs=-1           # use all CPU cores
# )

# rf_cv_scores = cross_val_score(rf_model, X_filled, y, cv=cv, scoring='roc_auc')
# print(f'Random Forest CV AUC: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')


### 5c. Gradient Boosting (LightGBM)

**Boosting** builds trees sequentially. Each new tree focuses on the mistakes the previous ones made. This often gives the best performance on tabular data. LightGBM is a fast implementation of gradient boosting.

Key differences from Random Forest:
- Trees are built **sequentially**, not independently
- LightGBM handles missing values **natively** (no need to fill NaNs!)
- Has a `learning_rate` that controls how aggressively each tree corrects errors

In [ ]:
# !pip install lightgbm  # uncomment if needed
# import lightgbm as lgb

# Hint: LightGBM can work directly with X (NaNs and all!)

# lgb_params = {
#     'objective': 'binary',
#     'metric': 'auc',
#     'learning_rate': ???,    # try 0.05 to 0.1
#     'num_leaves': ???,       # try 31, 63, 127
#     'verbose': -1,
# }

# Hint: for proper CV with LightGBM, you'll want to loop through folds manually
# and use lgb.Dataset + lgb.train with early_stopping.
# This prevents overfitting by stopping when the validation score stops improving.

# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
#     ...


### 5d. Your Own Model?

Try anything! XGBoost, CatBoost, a Neural Network, an ensemble of multiple models...

In [ ]:
# Go wild


## 7. Hyperparameter Tuning

Every model has **hyperparameters**. These r settings you choose before training (like `max_depth` or `learning_rate`). The right settings can significantly improve performance.

**GridSearchCV** tries every combination of parameters you specify and picks the best one via cross validation.

Here's an example with Logistic Regression. Apply the same idea to whichever model you're using.

In [ ]:
# GridSearchCV example with Logistic Regression
# 'C' controls regularisation - smaller C = more regularisation
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'liblinear']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=3,               # 3 folds to keep it fast
    scoring='roc_auc',
    verbose=1
)

grid_search.fit(X_scaled, y)

print(f'\nBest params: {grid_search.best_params_}')
print(f'Best AUC:    {grid_search.best_score_:.4f}')

### YOUR TURN - Tune Your Best Model

Apply hyperparameter tuning to whichever model gave you the best CV score. Some ideas:

- **Random Forest:** tune `n_estimators`, `max_depth`, `min_samples_split`
- **LightGBM:** tune `learning_rate`, `num_leaves`, `feature_fraction`, `reg_alpha`, `reg_lambda`

More combinations = longer runtime. Start small!

In [ ]:
# Your tuning here


## 8. Generate Submission

Submissions are evaluated using **ROC-AUC**, which measures how well your model ranks positive cases above negative ones. Because of this, you should submit **probabilities** (e.g. 0.73), not just binary 0/1 predictions.

The cell below uses the Logistic Regression baseline. **Replace `final_preds` with your best model's predictions.**

In [ ]:
# Replace this with your best model's predictions!
final_preds = lr_test_preds

submission = pd.DataFrame({
    'Id': test_ids,
    'Has_Valentine': final_preds
})

submission.to_csv('submission.csv', index=False)

print(f'Submission saved! Shape: {submission.shape}')
print(f'Prediction stats:')
print(f'  Min:  {final_preds.min():.4f}')
print(f'  Mean: {final_preds.mean():.4f}')
print(f'  Max:  {final_preds.max():.4f}')
print(f'\n{submission.head(10)}')
print(f'\nUpload submission.csv to Kaggle and see your score!')

## Ideas to Improve Your Score

Here are some things to try, roughly ordered from easiest to hardest:

- **Better features** - interaction terms (multiply two columns together), ratios, log transforms
- **Better models** - Random Forest, LightGBM, XGBoost, CatBoost
- **Better missing value handling** - instead of median, try different strategies per column
- **Hyperparameter tuning** - GridSearchCV, RandomizedSearchCV, or manual experimentation
- **Ensembling** - average predictions from multiple models (often the single biggest improvement)
- **Target encoding** - encode categoricals using the target variable (be careful of leakage!)
- **Feature selection** - remove noisy features that hurt performance

Good luck! 💘